# ETL Pipeline — Week 7

**Course:** CPSC 5071

**Team Name:** Team_5071_1

**Team:** Jordan Berke, Emily Larson, Jennifer Poling, Paul Skentzos

## Overview

We decided to add data that the Pacific Northwest National Laboratory (PNNL) collects about their sampling sites along rivers. They study how rivers behave over time. The goal is to compare these sampling sites with earthquake sites geographically, specifically, to identify which river monitoring stations are in close proximity to seismically active regions.

**External data source:** PNNL River Corridor & Watershed Biogeochemistry SFA — Geospatial Site Information  
**Download URL:** https://data.ess-dive.lbl.gov/view/doi:10.15485/1971251

**Why this source was chosen:**  
The PNNL dataset provides precise latitude/longitude coordinates for 681 river and stream monitoring sites across the United States. Pairing these with our earthquake catalog allows us to assess seismic risk near environmental monitoring infrastructure and reveals whether earthquake activity patterns cluster near watersheds of interest.

---
## Part 1 — Extract

In [1]:
# Import packages
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

In [2]:
# Load last week's cleaned earthquake data
earthquake = pd.read_csv('cleaned_data.csv')
# We need to clean our cleaned_data.csv to handle the event_year containing nulls.
# This was missed in last weeks project.
earthquake = earthquake.dropna(subset=['event_year'])
earthquake.head()

,location_id,latitude,longitude,depth_km,place_description,event_id,event_timestamp,last_updated,magnitude_value,measurement_error,num_stations_used,type_code,event_year,location,depth_category
0,46465,57.5699,-149.0911,27.8,"190 km E of Chiniak, Alaska",ak000126digo,2000-01-23 08:42:28.405000+00:00,2022-04-29 18:29:54.135000+00:00,5.5,0.1,27.0,mw,2000.0,"Chiniak, Alaska",Shallow
1,46438,65.0087,-154.2390,10.0,northern Alaska,ak0001kedehd,2000-02-03 10:24:57.773000+00:00,2022-04-29 18:30:52.396000+00:00,5.6,0.1,27.0,mw,2000.0,northern Alaska,Shallow
2,46359,60.2025,-145.9216,18.1,"39 km SSW of Cordova, Alaska",ak0002nyhqq3,2000-02-27 02:22:14.511000+00:00,2022-04-29 18:33:57.113000+00:00,5.0,0.1,27.0,mw,2000.0,"Cordova, Alaska",Shallow
3,46343,60.1896,-145.9005,10.0,"40 km S of Cordova, Alaska",ak0002t9i8pq,2000-03-01 23:05:14.227000+00:00,2022-04-29 19:23:12.999000+00:00,5.4,0.1,27.0,mw,2000.0,"Cordova, Alaska",Shallow
4,46328,57.3410,-154.2517,39.5,"27 km SW of Larsen Bay, Alaska",ak00034p02su,2000-03-08 14:20:57.427000+00:00,2022-04-29 18:34:34.872000+00:00,5.4,0.1,27.0,mw,2000.0,"Larsen Bay, Alaska",Shallow


In [3]:
# Load the new external data (PNNL river sampling sites)
# encoding='cp1252' handles Windows-1252 characters in the source file
site_info = pd.read_csv('v5_RCSFA_Geospatial_Site_Information.csv', encoding='cp1252')
site_info.head(5)

,Site_ID,Latitude,Longitude,IGSN,Description,Physiographic_Feature_Name,COMID,RawCOMID_Used,Country,Methods_and_Flags,Submission_Contact_Name,Submission_Contact_Email,Origin_Study_Code
0,ALAC,29.434045,-98.519132,10.58052/IEWDR022E,stream,Alazan Creek,10840404.0,True,United States,COMID_Analysis_01; COMID_01,James Stegen,james.stegen@pnnl.gov,AV1
1,Altamaha (Tidal driven),31.337603,-81.483730,10.58052/IEWDR028E,stream,Altamaha,14352998.0,True,United States,COMID_Analysis_01; COMID_02,James Stegen,james.stegen@pnnl.gov,48hr
2,APAC,29.414842,-98.514739,10.58052/IEWDR022C,stream,Apache Creek,10840420.0,True,United States,COMID_Analysis_01; COMID_01,James Stegen,james.stegen@pnnl.gov,AV1
3,BLU-BLU,44.162300,-122.332000,10.58052/IEWDR00WM,stream,Blue River,23773405.0,True,United States,COMID_Analysis_00; COMID_01,James Stegen,james.stegen@pnnl.gov,WROL
4,BLU-TID,44.217900,-122.265000,10.58052/IEWDR00WN,stream,Blue River below Tidbits Creek,23773429.0,True,United States,COMID_Analysis_00; COMID_02,James Stegen,james.stegen@pnnl.gov,WROL



## Part 2 — Transform

The transform phase covers:
1. Rename columns to consistent `snake_case` matching the earthquake dataset convention
2. Drop irrelevant columns (PII contact info, technical metadata flags)
3. Handle missing values
4. Add a `data_source` derived column to distinguish origin
5. Spatial nearest-earthquake join using a BallTree (Haversine distance)
6. Add a `proximity_category` derived column classifying seismic proximity

In [4]:
# Let's inspect the data types and missing values in the new source
print("=== site_info dtypes ===")
print(site_info.dtypes)
print("\n=== site_info null counts ===")
print(site_info.isnull().sum())

=== site_info dtypes ===
Site_ID                           str
Latitude                      float64
Longitude                     float64
IGSN                              str
Description                       str
Physiographic_Feature_Name        str
COMID                         float64
RawCOMID_Used                  object
Country                           str
Methods_and_Flags                 str
Submission_Contact_Name           str
Submission_Contact_Email          str
Origin_Study_Code                 str
dtype: object

=== site_info null counts ===
Site_ID                         0
Latitude                        0
Longitude                       0
IGSN                            0
Description                     0
Physiographic_Feature_Name      8
COMID                           0
RawCOMID_Used                 286
Country                         0
Methods_and_Flags               0
Submission_Contact_Name         0
Submission_Contact_Email        0
Origin_Study_Code           

In [5]:
# Step 1: Rename columns to consistent snake_case
site_clean = site_info.rename(columns={
    'Site_ID':                    'site_id',
    'Latitude':                   'latitude',
    'Longitude':                  'longitude',
    'IGSN':                       'igsn',
    'Description':                'description',
    'Physiographic_Feature_Name': 'feature_name',
    'COMID':                      'comid',
    'RawCOMID_Used':              'raw_comid_used',
    'Country':                    'country',
    'Methods_and_Flags':          'methods_flags',
    'Submission_Contact_Name':    'contact_name',
    'Submission_Contact_Email':   'contact_email',
    'Origin_Study_Code':          'origin_study_code'
})

print("Columns after rename:", site_clean.columns.tolist())

Columns after rename: ['site_id', 'latitude', 'longitude', 'igsn', 'description', 'feature_name', 'comid', 'raw_comid_used', 'country', 'methods_flags', 'contact_name', 'contact_email', 'origin_study_code']


In [6]:
# Step 2: Drop irrelevant columns
# - igsn: International Geo Sample Number — internal registry identifier, not needed for analysis
# - raw_comid_used: Boolean flag for internal PNNL processing — not analytically useful
# - contact_name / contact_email: PII — should not be propagated in the final dataset
# - methods_flags: Internal COMID analysis metadata — not relevant to spatial comparison

drop_cols = ['igsn', 'raw_comid_used', 'contact_name', 'contact_email', 'methods_flags']
site_clean = site_clean.drop(columns=drop_cols)

print(f"Shape after dropping columns: {site_clean.shape}")
print("Remaining columns:", site_clean.columns.tolist())

Shape after dropping columns: (681, 8)
Remaining columns: ['site_id', 'latitude', 'longitude', 'description', 'feature_name', 'comid', 'country', 'origin_study_code']


In [7]:
# Step 3: Handle missing values
# - feature_name (8 missing): fill with 'Unknown'
# - origin_study_code (20 missing): fill with 'Unknown'

site_clean['feature_name']      = site_clean['feature_name'].fillna('Unknown')
site_clean['origin_study_code'] = site_clean['origin_study_code'].fillna('Unknown')

print("Null counts after fill:")
print(site_clean.isnull().sum())

Null counts after fill:
site_id              0
latitude             0
longitude            0
description          0
feature_name         0
comid                0
country              0
origin_study_code    0
dtype: int64


In [8]:
# Step 4 (Derived Column 1): Add data_source identifier
# This distinguishes PNNL river sites from earthquake records in any downstream analysis
# that may combine both datasets.

site_clean['data_source'] = 'PNNL_River_Sampling'

print("data_source values:", site_clean['data_source'].unique())

data_source values: <StringArray>
['PNNL_River_Sampling']
Length: 1, dtype: str


### Step 5: Spatial Nearest-Earthquake Join

This step answers the core question of our integration: for each river sampling site,
which earthquake in our catalog occurred closest to it geographically?

The two datasets share no common key. The only thing they have in common is that both
contain latitude and longitude coordinates. This means we must join them spatially,
by proximity on the globe.

Latitude and longitude are angles, not distances. A one-degree difference in longitude
near the equator represents a much larger physical distance than the same difference near
the poles. To get a true ground distance between two points on Earth we use the
Haversine formula, which accounts for the Earth's curvature and returns a
great-circle distance

References used for BallTree:
- https://observablehq.com/@esperanc/omohundro-balltree-construction-algorithms
- https://www.astroml.org/book_figures/chapter2/fig_balltree_example.html
- https://towardsdatascience.com/k-nearest-neighbor-regressor-explained-a-visual-guide-with-code-examples-df5052c8c889/

In [9]:
# Step 5: Spatial nearest-earthquake join using sklearn BallTree (Haversine metric)
#
# For each PNNL river sampling site we find the single closest earthquake event
# in the earthquake dataset. BallTree with Haversine metric operates on
# (lat, lon) in radians and returns great-circle distances in radians,
# which we convert to kilometers (Earth radius ≈ 6,371 km).

# Build index over earthquake coordinates (convert to radians)
eq_coords_rad = np.radians(earthquake[['latitude', 'longitude']].values)
tree = BallTree(eq_coords_rad, metric='haversine')

# Query: find nearest earthquake for each river site
site_coords_rad = np.radians(site_clean[['latitude', 'longitude']].values)
dist_rad, idx   = tree.query(site_coords_rad, k=1)
dist_km         = dist_rad.flatten() * 6371.0 # 6371.0 = the radius of the earth in km

# Attach nearest earthquake attributes
site_clean['nearest_eq_distance_km']    = dist_km.round(2)
site_clean['nearest_eq_id']             = earthquake['event_id'].iloc[idx.flatten()].values
site_clean['nearest_eq_magnitude']      = earthquake['magnitude_value'].iloc[idx.flatten()].values
site_clean['nearest_eq_depth_km']       = earthquake['depth_km'].iloc[idx.flatten()].values
site_clean['nearest_eq_depth_category'] = earthquake['depth_category'].iloc[idx.flatten()].values
site_clean['nearest_eq_year']           = earthquake['event_year'].iloc[idx.flatten()].values
site_clean['nearest_eq_place']          = earthquake['place_description'].iloc[idx.flatten()].values

print(f"Spatial join complete. Nearest eq distance stats (km):")
print(site_clean['nearest_eq_distance_km'].describe().round(2))

Spatial join complete. Nearest eq distance stats (km):
count    681.00
mean     144.66
std      146.10
min        2.00
25%       46.84
50%       90.42
75%      180.34
max      834.54
Name: nearest_eq_distance_km, dtype: float64


### Step 6: Derived Column: proximity_category

Now that every river site has a `nearest_eq_distance_km` value, this step translates
that raw number into a human-readable category that is easier to filter, group, and
visualize in downstream analysis.

In [10]:
# Step 6 (Derived Column 2): Proximity category
# Classifies each river site by how close it is to the nearest recorded earthquake.
# Thresholds based on common seismological risk zones:
#   <100 km  — Very Close (potential direct ground motion impact)
#   100–500 km — Moderate
#   >500 km  — Distant

def proximity_category(distance_km):
    if distance_km < 100:
        return 'Very Close (<100 km)'
    elif distance_km < 500:
        return 'Moderate (100-500 km)'
    else:
        return 'Distant (>500 km)'

site_clean['proximity_category'] = site_clean['nearest_eq_distance_km'].apply(proximity_category)

print("Proximity category distribution:")
print(site_clean['proximity_category'].value_counts())

Proximity category distribution:
proximity_category
Very Close (<100 km)     372
Moderate (100-500 km)    280
Distant (>500 km)         29
Name: count, dtype: int64


In [11]:
# Let's do a Final check for verification: review shape, columns, and a sample
print(f"Final dataset shape: {site_clean.shape}")
print(f"Columns ({len(site_clean.columns)}): {site_clean.columns.tolist()}")
print("\nNull check:")
print(site_clean.isnull().sum())
print("\nData types:")
print(site_clean.dtypes)
print("\nSample rows:")
site_clean.head()

Final dataset shape: (681, 17)
Columns (17): ['site_id', 'latitude', 'longitude', 'description', 'feature_name', 'comid', 'country', 'origin_study_code', 'data_source', 'nearest_eq_distance_km', 'nearest_eq_id', 'nearest_eq_magnitude', 'nearest_eq_depth_km', 'nearest_eq_depth_category', 'nearest_eq_year', 'nearest_eq_place', 'proximity_category']

Null check:
site_id                      0
latitude                     0
longitude                    0
description                  0
feature_name                 0
comid                        0
country                      0
origin_study_code            0
data_source                  0
nearest_eq_distance_km       0
nearest_eq_id                0
nearest_eq_magnitude         0
nearest_eq_depth_km          0
nearest_eq_depth_category    0
nearest_eq_year              0
nearest_eq_place             0
proximity_category           0
dtype: int64

Data types:
site_id                          str
latitude                     float64
longitude  

,site_id,latitude,longitude,description,feature_name,comid,country,origin_study_code,data_source,nearest_eq_distance_km,nearest_eq_id,nearest_eq_magnitude,nearest_eq_depth_km,nearest_eq_depth_category,nearest_eq_year,nearest_eq_place,proximity_category
0,ALAC,29.434045,-98.519132,stream,Alazan Creek,10840404.0,United States,AV1,PNNL_River_Sampling,422.79,tx2024ophu,5.1,3.2959,Shallow,2024.0,"17 km NNE of Hermleigh, Texas",Moderate (100-500 km)
1,Altamaha (Tidal driven),31.337603,-81.483730,stream,Altamaha,14352998.0,United States,48hr,PNNL_River_Sampling,572.35,se60324281,5.1,4.1400,Shallow,2020.0,"4 km SE of Sparta, North Carolina",Distant (>500 km)
2,APAC,29.414842,-98.514739,stream,Apache Creek,10840420.0,United States,AV1,PNNL_River_Sampling,424.87,tx2024ophu,5.1,3.2959,Shallow,2024.0,"17 km NNE of Hermleigh, Texas",Moderate (100-500 km)
3,BLU-BLU,44.162300,-122.332000,stream,Blue River,23773405.0,United States,WROL,PNNL_River_Sampling,99.46,uw10306313,5.6,19.6080,Shallow,1993.0,"4 km E of Scotts Mills, Oregon",Very Close (<100 km)
4,BLU-TID,44.217900,-122.265000,stream,Blue River below Tidbits Creek,23773429.0,United States,WROL,PNNL_River_Sampling,94.81,uw10306313,5.6,19.6080,Shallow,1993.0,"4 km E of Scotts Mills, Oregon",Very Close (<100 km)


## Part 2 Complete — Transform Summary

The `site_clean` DataFrame is now a fully cleaned and enriched dataset of 681 river sampling sites × 17 columns. All column names are standardized to snake_case, irrelevant and PII columns have been removed, missing values filled, and each site has been spatially joined to its nearest earthquake event from the 106,077-record catalog, with distance, magnitude, depth, and a derived proximity category attached as new columns. This DataFrame is consistent and usable for analysis.

## Part 3 – Load 

In [12]:
## Step 1: Export complete enriched set

site_clean.to_csv('team7_final_data.csv', index=False)
print("Final dataset exported: team7_final_data.csv")

Final dataset exported: team7_final_data.csv


In [13]:
## Step 2: Saving each intermediate. 

# Save cleaned site data before spatial join
site_info.to_csv('team7_raw_pnnl_data_backup.csv', index=False)

# Save after cleaning but before spatial join
site_clean_before_join = site_clean.drop(columns=[
    'nearest_eq_distance_km',
    'nearest_eq_id',
    'nearest_eq_magnitude',
    'nearest_eq_depth_km',
    'nearest_eq_depth_category',
    'nearest_eq_year',
    'nearest_eq_place',
    'proximity_category'], errors='ignore')

site_clean_before_join.to_csv('team7_cleaned_sites_pre_join.csv', index=False)

# Save fully merged dataset
site_clean.to_csv('team7_sites_with_nearest_earthquake.csv', index=False)

print("All versions saved successfully.")

All versions saved successfully.
